# Generic (e.g., non-linear) filtering
stough 202-

In this brief demo we use [`scipy.ndimage.generic_filter`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.ndimage.generic_filter.html) to compute non-linear filters on an image, such as min/max/median and others. This works a lot more slowly than the `correlate` we used for the linear filters.

Every filter we've used so far has been *linear*: each output pixel is a fixed weighted sum of its neighborhood, which is why a single kernel of weights describes the whole operation. But there are useful neighborhood operations that can't be written as a weighted sum: the minimum or maximum of the neighborhood, its median, its range, and so on. These are **non-linear filters**. They still slide a window over the image and compute one output per pixel, but the computation can be any function of the pixels in the window.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np

# For computing a generic filter on an image
from scipy.ndimage import generic_filter

# For importing from alternative directory sources
import sys  
sys.path.insert(0, '../dip_utils')

from matrix_utils import (arr_info,
                          make_linmap)
from vis_utils import (vis_rgb_cube,
                       vis_hists,
                       vis_pair)

## Define the generic filtering function we want
This function will be applied to every *footprint* or neighborhood in the image. `generic_filter` hands the function a flat 1D array of the neighborhood's values (121 of them for an $11 \times 11$ window), and the function returns a single number: the output pixel.

In [ ]:
#simple blur
def mean_func(x):
    return x.mean()
#     xr = np.reshape(x, (5,5))
#     return np.mean(xr[:,3:])- np.mean(xr[:,:3])

def min_func(x):
    return x.min()

def max_func(x):
    return x.max()

def med_func(x):
    return np.median(x)

## Apply the function to each channel separately.
We could alternatively define the `footprint` or `size` to the `generic_filter` call to include the separate channels (e.g., `size = (3,3,3)`), but it's easier to think about this way.

We'll also normalize the image to float $[0,1]$ for visualization purposes.

In [ ]:
I = plt.imread('../dip_pics/bellagio.jpg').astype('float')
I = I/I.max()
arr_info(I)

In [ ]:
vis_hists(I)

In [ ]:
If = np.zeros(I.shape)
Out = np.zeros(I.shape[:2])

for chan in range(3):
    generic_filter(I[:, :, chan], function=mean_func, size=11, output=Out, mode='reflect')
    If[...,chan] = Out.copy();

In [ ]:
vis_pair(I, If, second_title="mean filtered")

The mean filter gives the same result as correlating with the $11 \times 11$ box kernel in the [low-pass demo](./imfilter_lowpass_demo.ipynb), but it takes much longer to compute. `correlate` runs its loop over pixels in compiled C code, while `generic_filter` has to call our Python function once for every pixel of every channel. That's millions of Python function calls for this image. (If you need a fast version of a common non-linear filter, `scipy.ndimage` has dedicated ones, such as `median_filter`, `minimum_filter` and `maximum_filter`.)

## Now try one of the other local neighborhood functions
The mean is in fact a linear operator, so we're not really looking into what these local non-linear operators can do for us. Try one of the others, like min or median, or even better, **write one of your own**. Show the same closeup for the different operators and consider why they look different.